# Beyond map graphs: Rome and Barabási–Albert graphs

*Generated by `utils/make_notebooks.py`. Edit the template there, not this file.*

The same six coloring algorithms, run on three families of graphs:

| family | what it is | in this notebook |
|---|---|---|
| **county** | US county maps, TIGER rook rule (shared border ≥ 100 m) | 52 maps (50 states + DC + PR) |
| **rome** | the Rome graph-drawing benchmark: sparse, mostly non-planar, 10–100 nodes | 1000 graphs, the same sample as `../GraphColoring` |
| **ba** | Barabási–Albert random graphs, `nx.barabasi_albert_graph(n, 3, seed)`; every new node attaches to 3 existing ones, so the mean degree is ≈ 6 and a few hubs get very high degree | 6 sizes × 5 seeds |

Sections:

1. What the three families look like (size, density, planarity, cliques)
2. Colors used vs. the chromatic number χ
3. Runtime vs. size
4. Does planarity matter? Rome graphs, planar vs. non-planar, compared at the same size
5. How hard the exact search works
6. A few sample graphs, colored by every algorithm

This notebook only **reads** results, it doesn't run the benchmark. It needs:

```
python -m utils.bench --no-maps --rome 1000 --ba-n 50 100 200 500 1000 2000 --ba-m 3 --ba-seeds 5 --repeats 5 --orders 10
python -m utils.graph_metrics --family rome
python -m utils.graph_metrics --family ba --probe-limit 10
python -m utils.graph_metrics                      # county_metrics.csv
```

Runtimes come from one machine. Compare them with each other, not with other machines.

In [ ]:
import os, sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

# Run from the repo root so `utils` and results/ resolve (see CLAUDE.md).
if Path.cwd().name == "stateNotebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import utils.bench as bench
import utils.color as wc

plt.rcParams["figure.dpi"] = 110
ALGOS = ["largest_first", "dsatur", "smallest_last", "independent_set", "exact", "sat"]
HEURISTICS = ALGOS[:4]
COUNTY_SOURCE = "tiger"   # "tiger" = rook rule (planar maps); "census" = Census adjacency (queen)

# one color + marker per family, used in every chart
FAMILIES = ["county", "rome", "ba"]
FAM_COLOR = {"county": "#2a78d6", "rome": "#eb6834", "ba": "#1baf7a"}
FAM_MARK = {"county": "o", "rome": "s", "ba": "^"}
INK, MUTED, GRID = "#2b2b2a", "#6b6b66", "#e4e3df"


def style(ax, xlabel="", ylabel="", title=""):
    ax.set_xlabel(xlabel, color=MUTED, fontsize=9); ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    ax.set_title(title, color=INK, fontsize=10, loc="left")
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8)


county = pd.read_csv("results/county_metrics.csv")
county = county[county["source"] == COUNTY_SOURCE].assign(family="county")
rome = pd.read_csv("results/rome_metrics.csv").assign(family="rome")
ba = pd.read_csv("results/ba_metrics.csv").assign(family="ba")
m = pd.concat([county, rome, ba], ignore_index=True)

runs = pd.read_csv("results/colorings.csv")
# colorings.csv says family "map" for counties; use this notebook's labels instead
runs = (runs[runs["graph"].isin(m["graph"])].drop(columns="family")
        .merge(m[["graph", "family", "planar"]], on="graph"))
print({f: int((m["family"] == f).sum()) for f in FAMILIES}, "graphs;", len(runs), "benchmark rows")

## 1. What the three families look like

The county maps and Rome graphs have about the same number of nodes, but Rome graphs are much
sparser (mean degree ≈ 2.6 vs. ≈ 5) and most of them are **not planar**. BA graphs with m = 3 are
the densest and the largest, and they contain hubs: a few nodes with very high degree.

In [ ]:
def summarize(g):
    return pd.Series({
        "graphs": len(g),
        "nodes (min-max)": f"{g['n'].min()}-{g['n'].max()}",
        "edges (median)": g["m"].median(),
        "mean degree (median)": round(g["mean_degree"].median(), 2),
        "max degree (median)": g["max_degree"].median(),
        "degeneracy (median)": g["degeneracy"].median(),
        "planar %": round(100 * g["planar"].mean()),
        "largest clique": dict(g["omega"].value_counts().sort_index()),
        "colors needed (chi)": dict(g["chi"].value_counts().sort_index()),
    })

pd.DataFrame({f: summarize(m[m["family"] == f]) for f in FAMILIES})

## 2. Colors used vs. the chromatic number

χ (the fewest colors possible) comes from `exact` and `sat`, which agree on every graph. A
heuristic "misses" a graph when it uses more than χ colors on the benchmark's fixed labeling.

In [ ]:
heur = runs[runs["algorithm"].isin(HEURISTICS)].copy()
heur["missed"] = heur["gap"] > 0
miss = heur.pivot_table(index="algorithm", columns="family", values="missed", aggfunc="mean")
miss = (100 * miss).loc[HEURISTICS, FAMILIES].round(1)
excess = heur.pivot_table(index="algorithm", columns="family", values="gap", aggfunc="max").loc[HEURISTICS, FAMILIES]

fig, ax = plt.subplots(figsize=(9, 4.2))
w = 0.26
for i, f in enumerate(FAMILIES):
    xs = np.arange(len(HEURISTICS)) + (i - 1) * w
    ax.bar(xs, miss[f], width=w - 0.03, color=FAM_COLOR[f], label=f, zorder=3)
    for x, v in zip(xs, miss[f]):
        ax.text(x, v + 1, f"{v:.0f}%", ha="center", fontsize=7.5, color=INK)
ax.set_xticks(range(len(HEURISTICS)), HEURISTICS)
style(ax, "", "% of graphs where it used more than chi colors",
      "How often each heuristic misses the minimum, by family")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

print("Most extra colors used on any one graph:")
display(excess)

## 3. Runtime vs. size

One panel per algorithm, log–log axes. On a log–log plot a straight line means a power law, and
its slope is the exponent: slope 1 is linear, slope 2 is quadratic. The fitted slope over all
families is in each title.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), sharex=True, sharey=True)
for ax, a in zip(axes.flat, ALGOS):
    r = runs[(runs["algorithm"] == a) & (runs["status"] != "timeout") & (runs["n"] >= 5)]
    for f in FAMILIES:
        d = r[r["family"] == f]
        ax.scatter(d["n"], d["runtime_ms_median"], s=12, alpha=0.55, color=FAM_COLOR[f],
                   marker=FAM_MARK[f], edgecolor="none", label=f)
    slope = np.polyfit(np.log(r["n"]), np.log(r["runtime_ms_median"]), 1)[0]
    ax.set_xscale("log"); ax.set_yscale("log")
    style(ax, "nodes (n)", "median ms", f"{a}  (runtime ~ n^{slope:.2f})")
axes.flat[0].legend(frameon=False, fontsize=8, markerscale=1.6)
plt.tight_layout(); plt.show()

med = runs.pivot_table(index="algorithm", columns="family", values="runtime_ms_median", aggfunc="median")
print("Median runtime per graph, ms:")
display(med.loc[ALGOS, FAMILIES].round(3))

## 4. Does planarity matter? Rome graphs at the same size

Most Rome graphs are non-planar, and the non-planar ones also tend to be larger and slightly
denser. Comparing all planar graphs with all non-planar graphs would mix up planarity with size.
So the Rome graphs are grouped into size bins, and planar and non-planar graphs are compared
**within each bin**.

In [ ]:
r = rome.copy()
r["size bin"] = pd.cut(r["n"], [0, 30, 50, 70, 100], labels=["10-30", "31-50", "51-70", "71-100"])
r["planarity"] = np.where(r["planar"], "planar", "non-planar")
for a in HEURISTICS:
    r[f"missed_{a}"] = r[f"gap_{a}"] > 0

agg = {"graphs": ("n", "size"), "mean degree": ("mean_degree", "mean")}
agg.update({f"{a} miss %": (f"missed_{a}", "mean") for a in HEURISTICS})
agg.update({"exact ms": ("ms_exact", "median"), "sat ms": ("ms_sat", "median")})
t = r.groupby(["size bin", "planarity"], observed=True).agg(**agg)
for a in HEURISTICS:
    t[f"{a} miss %"] = (100 * t[f"{a} miss %"]).round(1)
display(t.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
bins = list(t.index.get_level_values(0).unique())
for ax, a in zip(axes, ["largest_first", "independent_set"]):
    top = t[f"{a} miss %"].max()
    for i, (p, c) in enumerate([("planar", "#2a78d6"), ("non-planar", "#eb6834")]):
        vals = [t.loc[(b, p), f"{a} miss %"] if (b, p) in t.index else np.nan for b in bins]
        cnt = [int(t.loc[(b, p), "graphs"]) if (b, p) in t.index else 0 for b in bins]
        xs = np.arange(len(bins)) + (i - 0.5) * 0.38
        ax.bar(xs, vals, width=0.34, color=c, label=p, zorder=3)
        for x, v, k in zip(xs, vals, cnt):  # label each bar with how many graphs it averages
            text = "no graphs" if k == 0 else f"{v:.0f}%\n{k} graphs"
            ax.text(x, (0 if k == 0 else v) + top * 0.02, text, ha="center", va="bottom",
                    fontsize=7, color=INK if k else MUTED)
    ax.set_ylim(0, top * 1.25)
    ax.set_xticks(range(len(bins)), [f"{b} nodes" for b in bins])
    style(ax, "", "% of graphs missing chi", f"{a}: planar vs. non-planar Rome graphs, same size")
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

Within each size bin the non-planar bars are higher, but the *mean degree* column shows the
non-planar graphs are also **denser**. More edges per node means more constraints, and that
alone could explain the difference. So the next cell compares planar and non-planar graphs at
the same **density** (20–60 nodes, grouped by mean degree). It then fits a logistic regression
that predicts "missed χ" from planarity, size and density together. If planarity mattered on its
own, it would stay significant once density is in the model.

In [ ]:
d = r[r["n"].between(20, 60)].copy()
d["mean degree bin"] = pd.cut(d["mean_degree"], [0, 2.2, 2.4, 2.6, 2.8, 3.5])
for a in ["largest_first", "independent_set"]:
    g = d.groupby(["mean degree bin", "planarity"], observed=True)[f"missed_{a}"].agg(["mean", "size"])
    g["mean"] = (100 * g["mean"]).round(0)
    print(f"{a}: % missing chi (and number of graphs), same density")
    display(g.unstack("planarity").rename(columns={"mean": "miss %", "size": "graphs"}))

try:
    import statsmodels.formula.api as smf
except ImportError:
    print("pip install statsmodels to run the regression")
else:
    rr = rome[rome["omega"] < 4].copy()   # one graph has a 4-clique; it makes the fit unstable
    rr["planar"] = rr["planar"].astype(int)
    rows = {}
    for a in ["largest_first", "independent_set"]:
        rr["y"] = (rr[f"gap_{a}"] > 0).astype(int)
        alone = smf.logit("y ~ planar", data=rr).fit(disp=0)
        full = smf.logit("y ~ planar + n + mean_degree + max_degree + C(omega)", data=rr).fit(disp=0)
        rows[a] = {"planar effect alone": alone.params["planar"], "p (alone)": alone.pvalues["planar"],
                   "planar effect, controlling size+density": full.params["planar"],
                   "p (controlled)": full.pvalues["planar"],
                   "mean_degree effect": full.params["mean_degree"],
                   "p (mean_degree)": full.pvalues["mean_degree"]}
    display(pd.DataFrame(rows).T.style.format("{:.3g}"))

**How to read the regression:** a negative planar effect means planar graphs miss *less* often.
A p-value below 0.05 means the effect is unlikely to be chance. Planarity is a strong predictor
when it's the only variable. If its p-value jumps far above 0.05 once `n` and `mean_degree` are
included, then planarity only *looked* important because it travels with density and size.

## 5. How hard the exact search works

`exact` visits about one search-tree node per vertex when DSatur's order colors the graph
correctly on the first try. More nodes than that means backtracking. *Probe* columns rerun `exact` on
random vertex orders and report the worst order relative to the median.

In [ ]:
m["nodes per vertex"] = m["exact_nodes"] / m["n"].clip(lower=1)
m["no backtracking"] = m["exact_nodes"] <= m["n"] + m["k_steps"]
m["worst/median order"] = m["probe_nodes_max"] / m["probe_nodes_median"]
display(m.groupby("family").agg(**{
    "graphs": ("n", "size"),
    "no backtracking %": ("no backtracking", "mean"),
    "nodes per vertex (median)": ("nodes per vertex", "median"),
    "nodes per vertex (max)": ("nodes per vertex", "max"),
    "worst/median order (max)": ("worst/median order", "max"),
    "exact timeouts": ("exact_status", lambda s: int((s == "timeout").sum())),
    "probe timeouts": ("probe_timeouts", "sum"),
    "sat conflicts (max)": ("sat_conflicts", "max"),
}).loc[FAMILIES].round(2))

fig, ax = plt.subplots(figsize=(8, 4.5))
for f in FAMILIES:
    d = m[(m["family"] == f) & (m["exact_status"] == "ok") & (m["n"] >= 5)]
    ax.scatter(d["n"], d["nodes per vertex"], s=14, alpha=0.6, color=FAM_COLOR[f],
               marker=FAM_MARK[f], edgecolor="none", label=f)
ax.set_xscale("log"); ax.set_yscale("log")
style(ax, "nodes (n)", "search-tree nodes per vertex (1 = no backtracking)",
      "Exact search effort by family")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 6. Sample graphs, colored by every algorithm

`SAMPLE` picks a few graphs: two Rome graphs of similar size (one planar, one not) and two BA
graphs. Edit the list to look at others. A Rome graph is named by its file (`grafo<id>.<n>`), and
a BA graph by `(n, seed)`. Each graph is relabeled with `integer_labels` and colored the same way
as in the benchmark, so the color counts match `results/colorings.csv`.

In [ ]:
def pick_rome(planar):
    """A ~50-node Rome graph where largest_first misses chi, if one exists."""
    d = rome[(rome["planar"] == planar) & rome["n"].between(40, 60)]
    hard = d[d["gap_largest_first"] > 0]
    return (hard if len(hard) else d).sort_values("graph").iloc[0]["graph"].removeprefix("rome_")

SAMPLE = [("rome", pick_rome(True)), ("rome", pick_rome(False)), ("ba", (50, 0)), ("ba", (200, 0))]
SAMPLE

In [ ]:
EXTRA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948",
         "#7f7f7f", "#000000"]  # enough distinct colors for up to 10 classes


def load_sample(kind, key):
    if kind == "rome":
        path = next(p for p in bench.rome_sample(1000) if p.stem == key)
        G = nx.Graph(nx.read_graphml(path)); G.remove_edges_from(list(nx.selfloop_edges(G)))
        return f"Rome {key}", G
    n, seed = key
    return f"BA n={n}, m=3, seed={seed}", nx.barabasi_albert_graph(n, 3, seed=seed)


def color_all(G):
    out = {name: nx.greedy_color(G, strategy=s) for name, s in wc.ALGORITHMS.items()}
    out["exact"] = wc.exact_coloring(G, time_limit=60)[0]
    out["sat"] = wc.sat_coloring(G, time_limit=60)[0]
    for a, col in out.items():
        assert not wc.is_proper(G, col), a
    return out


table = []
for kind, key in SAMPLE:
    title, G = load_sample(kind, key)
    G = wc.integer_labels(G)
    cols = color_all(G)
    chi = len(set(cols["exact"].values()))
    pos = nx.kamada_kawai_layout(G) if G.number_of_nodes() <= 120 else nx.spring_layout(G, seed=0)
    hub_size = [20 + 6 * d for _, d in G.degree] if kind == "ba" else 60

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, a in zip(axes.flat, ALGOS):
        k = len(set(cols[a].values()))
        nx.draw_networkx_edges(G, pos, ax=ax, width=0.5, edge_color="#bdbdb8")
        nx.draw_networkx_nodes(G, pos, ax=ax, node_size=hub_size, linewidths=0.6, edgecolors="white",
                               node_color=[EXTRA[cols[a][v] % len(EXTRA)] for v in G])
        extra = f"  (+{k - chi} over chi)" if k > chi else ""
        ax.set_title(f"{a}: {k} colors{extra}", fontsize=10, loc="left", color=INK)
        ax.set_axis_off()
    fig.suptitle(f"{title}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
                 f"planar={nx.check_planarity(G)[0]}, chi={chi}", x=0.01, ha="left", fontsize=12)
    plt.tight_layout(); plt.show()
    table.append({"graph": title, "n": G.number_of_nodes(), "m": G.number_of_edges(),
                  "largest clique": max(map(len, nx.find_cliques(G))), "chi": chi,
                  **{a: len(set(cols[a].values())) for a in ALGOS}})

pd.DataFrame(table).set_index("graph")

In the BA drawings, node size is proportional to degree, so the hubs stand out. Hubs are the
reason `largest_first` and `independent_set` do worse on BA graphs: a hub's many neighbours make
the greedy choices early on matter a lot.